In [0]:
%pip install great_expectations

In [0]:
import pandas as pd
import great_expectations as gx
from great_expectations.expectations import (
    ExpectColumnValuesToNotBeNull,
    ExpectColumnValuesToBeInSet,
    ExpectColumnValuesToBeBetween,
    ExpectColumnValuesToMatchRegex,
    ExpectColumnValueLengthsToBeBetween,
)

dbutils.widgets.text("cleaned_data_file", "/Volumes/workspace/hdb/hdb-output-data/hdb_cleaned_data.csv")

cleaned_data_file = dbutils.widgets.get("cleaned_data_file")


In [0]:

# Initialize the data context
context = gx.get_context()

# Load your dataset into a pandas DataFrame
df = pd.read_csv(cleaned_data_file, dtype={"block": str})

# Connect the DataFrame to Great Expectations as a Data Asset
data_source = context.data_sources.add_pandas("pandas_datasource")

# Add asset and batch definition
data_asset = data_source.add_dataframe_asset(name="hdb_asset")
batch_definition = data_asset.add_batch_definition_whole_dataframe("hdb_batch")

# Fetch the batch
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})

# create suit
suite = gx.ExpectationSuite(name="hdb_resale_suite")

In [0]:
# Add expectations to the suite
suite.add_expectation(ExpectColumnValuesToNotBeNull(column="town"))
suite.add_expectation(ExpectColumnValuesToBeInSet(column="town", value_set=['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH','BUKIT TIMAH', 'CENTRAL AREA', 'CHOA CHU KANG', 'CLEMENTI',
       'GEYLANG', 'HOUGANG', 'JURONG EAST', 'JURONG WEST',
       'KALLANG/WHAMPOA', 'MARINE PARADE', 'QUEENSTOWN', 'SENGKANG',
       'SERANGOON', 'TAMPINES', 'TOA PAYOH', 'WOODLANDS', 'YISHUN',
       'LIM CHU KANG', 'SEMBAWANG', 'BUKIT PANJANG', 'PASIR RIS',
       'PUNGGOL']))

suite.add_expectation(ExpectColumnValuesToNotBeNull(column="flat_type"))
suite.add_expectation(ExpectColumnValuesToBeInSet(column="flat_type", value_set=['1 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', '2 ROOM', 'EXECUTIVE','MULTI GENERATION']))

suite.add_expectation(ExpectColumnValuesToNotBeNull(column="flat_model"))
suite.add_expectation(ExpectColumnValuesToBeInSet(column="flat_model", value_set=['IMPROVED', 'NEW GENERATION', 'MODEL A', 'STANDARD', 'SIMPLIFIED',
       'MODEL A MAISONETTE', 'APARTMENT', 'MAISONETTE', 'TERRACE',
       '2 ROOM', 'IMPROVED MAISONETTE', 'MULTI GENERATION',
       'PREMIUM APARTMENT', 'ADJOINED FLAT', 'PREMIUM MAISONETTE',
       'MODEL A2', 'TYPE S1', 'TYPE S2', 'DBSS', 'PREMIUM APARTMENT LOFT',
       '3GEN']))




In [0]:
suite.add_expectation(ExpectColumnValuesToNotBeNull(column="month"))
suite.add_expectation(ExpectColumnValuesToMatchRegex(column="month", regex=r"^\d{4}-(0[1-9]|1[0-2])$"))

suite.add_expectation(ExpectColumnValuesToNotBeNull(column="storey_range"))
suite.add_expectation(ExpectColumnValuesToMatchRegex(column="storey_range", regex=r"^\d{2}\s+TO\s+\d{2}$"))

suite.add_expectation(ExpectColumnValuesToNotBeNull(column="block"))
suite.add_expectation(ExpectColumnValuesToMatchRegex(column="block", regex=r"^[0-9]{3}$"))


In [0]:
# Run Validation
results = batch.validate(suite)

# Check overall pass/fail status
print(f"Validation Success: {results.success}\n")

# Print details of any failed expectations
if not results.success:
    for res in results.results:
        if not res.success:
            print(f"Failed Column: {res.expectation_config.kwargs.get('column')}")
            print(f"Expectation: {res.expectation_config.type}")
            print(f"Details: {res.result}\n")